# My task write-up: Implementing the production navigation inference endpoint

**Task:** Add the production-facing ML inference path that serves our approved WalkBuddy navigation model through the existing backend. Scope was *only* ML model serving and its API contract — no auth, routing, chat, OCR, or speech.

**What I built (in my own words):** I added a brand-new endpoint, `POST /ml/navigate`, that takes a camera frame, runs it through the YOLO model we already load at startup, and returns the detections plus a spoken-style guidance message in a clean, documented JSON shape. I also added a small `GET /ml/model-info` so the frontend can ask which classes the model knows. On top of that I added a **mock mode** (an env flag) so my teammates on the frontend can build against the API *before* a real navigation model exists.

The most important rule I gave myself was: **this is a pure addition**. I did not touch `/vision`, `/ws/vision`, or any other existing route. I reused the model, the adapter, and the helper functions that were already there instead of writing new inference logic.

Files I added or changed:

- `routers/ml_inference.py` — **new**, the two endpoints + mock mode.
- `main.py` — **changed**, 2 lines to register the new router.
- `tests/test_ml_inference.py` — **new**, 9 tests that don't need real weights.
- `requirements-dev.txt` — **new**, so the tests run in CI.
- `docs/ml_inference.md` — **new**, human-readable docs for the endpoint.

## 1. What I found in the existing backend before writing any code

Before touching anything, I traced how vision already works so I could reuse it instead of reinventing it. Here's what I found:

- **The model is loaded once at startup.** In `main.py` there's a `lifespan` function that runs when the server boots. It does `app.state.yolo = YOLO(str(YOLO_MODEL_PATH))`. So the model lives on `app.state.yolo` and every request can grab it from there — I don't need to load it myself. The path comes from an env var `WALKBUDDY_MODEL_DIR` (for Docker) or falls back to `ML_side/models/best.pt`. If loading fails it sets `app.state.yolo = None` instead of crashing.
- **There's a concurrency limiter.** `app.state.vision_limiter` is an `anyio.CapacityLimiter(1)`, which means only one YOLO inference runs at a time. The existing `/vision` endpoint uses it, so I use it too to stay consistent.
- **`/vision` and `/ws/vision` both call the same `vision_adapter`.** The real detection logic lives in `adapters/vision_adapter.py`. It runs `model.predict(...)` and returns a dict. I found the exact shape it returns: an `image_id`, a list of `detections` (each with `category`, `confidence`, `bbox`, `direction`, `priority`), and `metadata` with the image shape. The class *name* for each detection comes straight from the model itself (`result.names[cls_id]`) — it's **not** hardcoded anywhere.
- **Guidance messages are built by helpers.** `routers/ai_service.py` has `_event_from_detection(...)` and `_guidance_payload(...)`. The second one first asks the safety gate if anything dangerous is ahead, and otherwise turns detections into a short message like "table ahead". I decided to reuse both so my endpoint gives the same guidance as the existing ones.

**The one thing that surprised me (the 7-vs-8 class discrepancy):** the training config `ML_side/config/newdata.yaml` lists **8** classes (it adds `couch`), but the docs for the actual `best.pt` file (`ML_side/models/README.md`) say the verified weights only have **7** classes. So the config and the real model file don't match. I decided **not** to hardcode either list — instead I read the class names from the loaded model at runtime, so my code is correct no matter which weights are loaded. Fixing the lineage mismatch is someone else's follow-up job.

The code cell below is the existing code I studied (copied from `main.py` and `vision_adapter.py`) — this is what I built on top of, not code I wrote.

In [ ]:
# --- EXISTING CODE I STUDIED (not mine) ---

# main.py — the model is loaded once at startup and stored on app.state
#     try:
#         logger.info(f"Loading YOLO from {YOLO_MODEL_PATH}")
#         app.state.yolo = YOLO(str(YOLO_MODEL_PATH))
#         logger.info("\u2705 YOLO ready")
#     except Exception as e:
#         logger.error(f"\u274c YOLO load failed: {e}")
#         app.state.yolo = None
#     ...
#     app.state.vision_limiter = anyio.CapacityLimiter(1)   # one inference at a time


# adapters/vision_adapter.py — the detection logic + the dict shape it returns
def vision_adapter(model, image_path):
    results = model.predict(source=image_path, conf=0.25, iou=0.45, verbose=False)
    result = results[0]
    detections = []
    # ... reads image size, then for each box: ...
    #     label = result.names[cls_id]        # <-- class NAME comes from the model itself
    #     direction = calculate_spatial_position(bbox, image_width)  # left / right / ahead
    #     priority = get_priority(label)      # HIGH / MEDIUM / LOW
    #     detections.append({
    #         "category": label,
    #         "confidence": round(conf, 3),
    #         "bbox": {"x_min": ..., "y_min": ..., "x_max": ..., "y_max": ...},
    #         "direction": direction,
    #         "priority": priority,
    #     })
    return {
        "image_id": "<file stem>",
        "detections": detections,
        "metadata": {"image_shape": [image_height, image_width]},
    }


# routers/ai_service.py — the helpers I reused for guidance + memory
def _event_from_detection(detection):
    return {
        "label": detection["category"],
        "direction": detection.get("direction", "ahead"),
        "distance_m": None,
        "confidence": detection["confidence"],
        # ... plus motion fields (track_id, is_moving, approaching, ...)
    }

def _guidance_payload(result, max_messages=1):
    # 1) ask the safety gate for a STOP message if a hazard is ahead
    # 2) otherwise build a short spoken message from the detections
    # returns (message_string, risk_level_string)
    ...


## 2. File I added: `routers/ml_inference.py`

This is the main file — the actual endpoint. Here's how it works and why I wrote it the way I did:

**Why a new file?** I asked myself whether to bolt this onto the existing `ai_service.py` or make a new router. I went with a new file `routers/ml_inference.py` because the task is specifically about *production ML serving*, and keeping it separate makes the scope obvious and easy to review. It uses `APIRouter(prefix="/ml")` so both endpoints live under `/ml`.

**How `POST /ml/navigate` works:**
1. If mock mode is on, it returns a fixed fake result and stops (more on that below).
2. It grabs the already-loaded model from `request.app.state.yolo`. If that's `None`, it returns **503** — exactly like the existing `/vision` does.
3. It reads the class list from the model with my little `_model_classes()` helper (explained below).
4. If the uploaded file is empty, it short-circuits with an empty-but-valid response.
5. Otherwise it writes the frame to a temp file, waits for the shared `vision_limiter` slot, and runs `vision_adapter` in a worker thread (so it doesn't block the event loop). This is the same pattern the existing code uses.
6. It saves each detection into `state.memory` (so chat/LLM context stays in sync) and builds the guidance message with `_guidance_payload`.
7. It returns a tidy JSON contract: `model`, `classes`, `detections`, `guidance_message`, `risk_level`, `inference_time_ms`, `image_id`.
8. A `finally` block always deletes the temp file, even if inference blows up. If the adapter raises, it returns **500** — again matching the existing behaviour.

**Reading classes from the model (not hardcoding):** `_model_classes()` reads `yolo.names`. Ultralytics gives `names` as a dict like `{0: "book", 1: "books", ...}`, so I sort by the integer key to get the right order, and I also handle the case where it's a plain list. This is the bit that makes the endpoint work for **both** the 7-class and 8-class weights — I never wrote the class list into my code. `model` is just a label derived from the count (e.g. `walkbuddy-yolo-8class`).

**Mock mode (`WALKBUDDY_ML_MOCK`):** the task said development can start with mocked predictions. So I added `_mock_enabled()` which checks an env flag (off by default). When it's on, both endpoints return a **deterministic** fake detection (a `table` ahead) in the **exact same contract**, and — importantly — it works even when no weights are loaded (`app.state.yolo` can be `None`). The `model` field becomes `walkbuddy-yolo-mock` so the frontend can tell it's fake data. I still run the real `_guidance_payload` on the fake detection so the guidance text is realistic.

The full file is below.

In [ ]:
"""
Production ML inference endpoint for the WalkBuddy navigation model.

This router is a *pure addition* on top of the existing vision pipeline. It
reuses the already-loaded YOLO model (`app.state.yolo`), the shared capacity
limiter (`app.state.vision_limiter`), the `vision_adapter` detection logic, the
`_event_from_detection` / `_guidance_payload` helpers, and `state.memory`. It
introduces no new inference logic and does not touch `/vision`, `/ws/vision`,
or any other route.

The `model` and `classes` fields are read from `app.state.yolo.names` at request
time, so the contract works unchanged for both the 7-class and 8-class weights
without hardcoding the taxonomy. (See `ML_side/models/README.md` and
`ML_side/docs/current_model_baseline.md` for the known best.pt vs. newdata.yaml
class-lineage discrepancy — resolving that is separate follow-up work.)

Mock mode: set `WALKBUDDY_ML_MOCK=1` (default off) to make both endpoints return
a deterministic fake result in the exact same contract, with no weights and no
inference. This lets the frontend / API be developed before a navigation model
exists. The mock reports the approved eight-class taxonomy.
"""

import os
import time
import tempfile
import logging

import anyio
from fastapi import APIRouter, UploadFile, File, Request, HTTPException

from adapters.vision_adapter import vision_adapter
from internal import state
from routers.ai_service import _event_from_detection, _guidance_payload

logger = logging.getLogger(__name__)
router = APIRouter(prefix="/ml", tags=["ml"])

# ── Mock mode ───────────────────────────────────────────────────────────────
# When WALKBUDDY_ML_MOCK is truthy the endpoints return a deterministic fake
# result (no weights, no inference) so the API can be developed before a real
# navigation model exists. Off by default.
_TRUTHY = {"1", "true", "yes", "on"}

# Approved eight-class navigation taxonomy (matches ML_side/config/newdata.yaml).
MOCK_CLASSES = [
    "book", "books", "monitor", "office-chair",
    "whiteboard", "table", "tv", "couch",
]
MOCK_MODEL = "walkbuddy-yolo-mock"

# A single deterministic detection in the exact shape vision_adapter produces.
_MOCK_RESULT = {
    "image_id": "mock",
    "detections": [
        {
            "category": "table",
            "confidence": 0.87,
            "bbox": {"x_min": 220, "y_min": 180, "x_max": 420, "y_max": 400},
            "direction": "ahead",
            "priority": "HIGH",
        }
    ],
    "metadata": {"image_shape": [480, 640]},
}


def _mock_enabled() -> bool:
    return os.getenv("WALKBUDDY_ML_MOCK", "").strip().lower() in _TRUTHY


def _model_classes(yolo) -> list[str]:
    """Return the model's class names in class-index order.

    Reads directly from the loaded model's `.names` so the contract reflects the
    actual weights (7- or 8-class) rather than a hardcoded list. Ultralytics
    exposes `.names` as a dict keyed by int class id; a plain list is also
    handled defensively.
    """
    names = getattr(yolo, "names", None)
    if names is None:
        return []
    if isinstance(names, dict):
        return [names[key] for key in sorted(names)]
    return list(names)


def _model_descriptor(classes: list[str]) -> str:
    return f"walkbuddy-yolo-{len(classes)}class"


@router.get("/model-info")
async def model_info(request: Request):
    """Expose the authoritative class list/order baked into the active weights."""
    if _mock_enabled():
        return {
            "model": MOCK_MODEL,
            "classes": list(MOCK_CLASSES),
            "class_count": len(MOCK_CLASSES),
            "mock": True,
        }

    yolo = request.app.state.yolo
    if yolo is None:
        raise HTTPException(503, "Vision model unavailable")

    classes = _model_classes(yolo)
    return {
        "model": _model_descriptor(classes),
        "classes": classes,
        "class_count": len(classes),
    }


@router.post("/navigate")
async def navigate_endpoint(request: Request, file: UploadFile = File(...)):
    """Run the approved navigation model on a single frame.

    Contract (200):
        {
          "model": str,                 # derived from app.state.yolo.names
          "classes": list[str],         # class names in index order
          "detections": list[dict],     # exactly what vision_adapter returns
          "guidance_message": str,
          "risk_level": str,
          "inference_time_ms": int,
          "image_id": str | None,
        }
    """
    if _mock_enabled():
        result = _MOCK_RESULT
        for d in result["detections"]:
            state.memory.add_event(**_event_from_detection(d))
        guidance, risk_level = _guidance_payload(result, max_messages=3)
        return {
            "model": MOCK_MODEL,
            "classes": list(MOCK_CLASSES),
            "detections": result["detections"],
            "guidance_message": guidance,
            "risk_level": risk_level,
            "inference_time_ms": 0,
            "image_id": result["image_id"],
        }

    yolo = request.app.state.yolo
    if yolo is None:
        raise HTTPException(503, "Vision model unavailable")

    classes = _model_classes(yolo)

    content = await file.read()
    if not content:
        return {
            "model": _model_descriptor(classes),
            "classes": classes,
            "detections": [],
            "guidance_message": "",
            "risk_level": "CLEAR",
            "inference_time_ms": 0,
            "image_id": None,
        }

    temp_path = None
    try:
        suffix = os.path.splitext(file.filename or "frame.jpg")[1] or ".jpg"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as f:
            f.write(content)
            temp_path = f.name

        t0 = time.monotonic()
        try:
            async with request.app.state.vision_limiter:
                result = await anyio.to_thread.run_sync(
                    vision_adapter,
                    yolo,
                    temp_path,
                )
        except Exception as e:
            logger.error(f"ML navigate adapter error: {e}")
            raise HTTPException(500, "Vision processing failed")
        inference_ms = int((time.monotonic() - t0) * 1000)

        for d in result["detections"]:
            state.memory.add_event(**_event_from_detection(d))

        guidance, risk_level = _guidance_payload(result, max_messages=3)

        return {
            "model": _model_descriptor(classes),
            "classes": classes,
            "detections": result["detections"],
            "guidance_message": guidance,
            "risk_level": risk_level,
            "inference_time_ms": inference_ms,
            "image_id": result["image_id"],
        }

    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)


## 3. File I changed: `main.py` (only 2 lines)

To keep this a pure addition, I only added **two lines** to `main.py`: one to import my new router, and one to register it on the app. I put them right next to where the existing routers are imported and included, so it follows the same pattern as everything else. I didn't touch the startup code, the middleware, or any existing route.

In [ ]:
# In the "Routers" import block:
from routers import ai_service as ai_router
from routers import ml_inference as ml_router     # <-- LINE 1 I ADDED
from routers import helpers as helpers_router

# In the "8. ROUTERS" section where routers are registered:
app.include_router(audiobooks_router.router)
app.include_router(ai_router.router)
app.include_router(ml_router.router)              # <-- LINE 2 I ADDED
app.include_router(helpers_router.router)


## 4. File I added: `tests/test_ml_inference.py`

I wanted tests that prove the endpoint works **without downloading the real model weights** (weights aren't in git, and CI won't have them). So the trick I used is: I build a tiny throwaway FastAPI app in the test, mount *only* my router on it, and set `app.state.yolo` to a fake object that just has a `.names` dict. For the tests that need "inference", I `monkeypatch` `vision_adapter` to return a canned result. That way I'm testing my endpoint's logic and contract, not YOLO itself.

Here's what each test checks:

- **`test_navigate_contract_eight_class`** — the happy path with 8-class weights. Checks the response has *exactly* the 7 contract keys (nothing missing/extra), that `classes` and `model` came from the fake model's `.names` (so `walkbuddy-yolo-8class`), and that the detections pass through unchanged.
- **`test_navigate_contract_seven_class`** — the *same* endpoint with 7-class weights. Proves I didn't hardcode the class list: `model` becomes `walkbuddy-yolo-7class` and `couch` is not in the list.
- **`test_navigate_empty_file_short_circuits`** — if someone uploads an empty file, it returns a valid empty response (no crash, `image_id` is null).
- **`test_navigate_503_when_model_unavailable`** — if `app.state.yolo` is `None`, it returns 503 with the right message.
- **`test_navigate_500_on_adapter_error`** — if the adapter throws, it returns 500 with the right message (and the temp file still gets cleaned up).
- **`test_model_info_reports_active_classes`** — `GET /ml/model-info` reports the active class list and count.
- **`test_model_info_503_when_model_unavailable`** — model-info also returns 503 when no model.
- **`test_navigate_mock_mode_no_weights`** — turns the env flag on with `yolo=None` and proves mock mode returns the deterministic fake in the exact same contract, *with no weights loaded*.
- **`test_model_info_mock_mode_no_weights`** — same idea for model-info, and checks the `"mock": true` flag.

The full test file is below.

In [ ]:
"""
Unit tests for the production ML inference endpoint (routers/ml_inference.py).

These tests never load real model weights. They mount only the ml_inference
router on a bare FastAPI app, set a fake `app.state.yolo` (with a `.names`
mapping) plus a real capacity limiter, and stub out `vision_adapter` so the
contract can be verified without inference.

Run from the backend directory:
    pytest tests/test_ml_inference.py -v
"""

import anyio
import pytest
from fastapi import FastAPI
from fastapi.testclient import TestClient

import routers.ml_inference as ml_inference


SEVEN_CLASSES = {
    0: "book",
    1: "books",
    2: "monitor",
    3: "office-chair",
    4: "whiteboard",
    5: "table",
    6: "tv",
}

EIGHT_CLASSES = {**SEVEN_CLASSES, 7: "couch"}


class FakeYolo:
    """Minimal stand-in for an Ultralytics YOLO model.

    Only `.names` is read by the endpoint; inference is stubbed separately.
    """

    def __init__(self, names):
        self.names = names


def _fake_result():
    return {
        "image_id": "frame",
        "detections": [
            {
                "category": "table",
                "confidence": 0.91,
                "bbox": {"x_min": 100, "y_min": 120, "x_max": 400, "y_max": 460},
                "direction": "ahead",
                "priority": "HIGH",
            }
        ],
        "metadata": {"image_shape": [480, 640]},
    }


def _build_app(yolo):
    app = FastAPI()
    app.include_router(ml_inference.router)
    app.state.yolo = yolo
    app.state.vision_limiter = anyio.CapacityLimiter(1)
    return app


@pytest.fixture
def stub_adapter(monkeypatch):
    """Replace vision_adapter with a canned result (no weights, no inference)."""
    monkeypatch.setattr(ml_inference, "vision_adapter", lambda model, path: _fake_result())


# ---------------------------------------------------------------------------
# /ml/navigate — contract
# ---------------------------------------------------------------------------

def test_navigate_contract_eight_class(stub_adapter):
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()

    # Exact contract keys, nothing missing.
    assert set(body) == {
        "model",
        "classes",
        "detections",
        "guidance_message",
        "risk_level",
        "inference_time_ms",
        "image_id",
    }

    # classes + model are derived from app.state.yolo.names (not hardcoded).
    assert body["classes"] == [
        "book", "books", "monitor", "office-chair",
        "whiteboard", "table", "tv", "couch",
    ]
    assert body["model"] == "walkbuddy-yolo-8class"

    # Detections pass through verbatim from vision_adapter.
    assert body["detections"] == _fake_result()["detections"]
    assert body["image_id"] == "frame"
    assert isinstance(body["inference_time_ms"], int)
    assert isinstance(body["guidance_message"], str)
    assert isinstance(body["risk_level"], str) and body["risk_level"]


def test_navigate_contract_seven_class(stub_adapter):
    """Same endpoint must work unchanged for the 7-class weights."""
    client = TestClient(_build_app(FakeYolo(SEVEN_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()
    assert body["model"] == "walkbuddy-yolo-7class"
    assert body["classes"] == [
        "book", "books", "monitor", "office-chair",
        "whiteboard", "table", "tv",
    ]
    assert "couch" not in body["classes"]


def test_navigate_empty_file_short_circuits(stub_adapter):
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()
    assert body["detections"] == []
    assert body["image_id"] is None
    assert body["model"] == "walkbuddy-yolo-8class"
    assert body["classes"][0] == "book"


# ---------------------------------------------------------------------------
# /ml/navigate — 503 when model unavailable
# ---------------------------------------------------------------------------

def test_navigate_503_when_model_unavailable():
    client = TestClient(_build_app(yolo=None))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 503
    assert resp.json()["detail"] == "Vision model unavailable"


# ---------------------------------------------------------------------------
# /ml/navigate — 500 on inference failure (temp file still cleaned up)
# ---------------------------------------------------------------------------

def test_navigate_500_on_adapter_error(monkeypatch):
    def _boom(model, path):
        raise RuntimeError("cuda exploded")

    monkeypatch.setattr(ml_inference, "vision_adapter", _boom)
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"not-a-real-jpeg", "image/jpeg")},
    )
    assert resp.status_code == 500
    assert resp.json()["detail"] == "Vision processing failed"


# ---------------------------------------------------------------------------
# /ml/model-info
# ---------------------------------------------------------------------------

def test_model_info_reports_active_classes():
    client = TestClient(_build_app(FakeYolo(EIGHT_CLASSES)))
    resp = client.get("/ml/model-info")
    assert resp.status_code == 200
    body = resp.json()
    assert body["model"] == "walkbuddy-yolo-8class"
    assert body["class_count"] == 8
    assert body["classes"][-1] == "couch"


def test_model_info_503_when_model_unavailable():
    client = TestClient(_build_app(yolo=None))
    resp = client.get("/ml/model-info")
    assert resp.status_code == 503


# ---------------------------------------------------------------------------
# Mock mode (WALKBUDDY_ML_MOCK) — works with NO weights loaded
# ---------------------------------------------------------------------------

def test_navigate_mock_mode_no_weights(monkeypatch):
    monkeypatch.setenv("WALKBUDDY_ML_MOCK", "1")
    # yolo is None on purpose: mock mode must not require weights.
    client = TestClient(_build_app(yolo=None))
    resp = client.post(
        "/ml/navigate",
        files={"file": ("frame.jpg", b"ignored-in-mock", "image/jpeg")},
    )
    assert resp.status_code == 200
    body = resp.json()

    # Exact same contract as the real path.
    assert set(body) == {
        "model",
        "classes",
        "detections",
        "guidance_message",
        "risk_level",
        "inference_time_ms",
        "image_id",
    }

    # Deterministic mock payload, approved eight-class taxonomy.
    assert body["model"] == "walkbuddy-yolo-mock"
    assert body["classes"] == [
        "book", "books", "monitor", "office-chair",
        "whiteboard", "table", "tv", "couch",
    ]
    assert body["detections"] == [
        {
            "category": "table",
            "confidence": 0.87,
            "bbox": {"x_min": 220, "y_min": 180, "x_max": 420, "y_max": 400},
            "direction": "ahead",
            "priority": "HIGH",
        }
    ]
    assert body["image_id"] == "mock"
    assert body["inference_time_ms"] == 0
    assert isinstance(body["guidance_message"], str) and body["guidance_message"]
    assert isinstance(body["risk_level"], str) and body["risk_level"]


def test_model_info_mock_mode_no_weights(monkeypatch):
    monkeypatch.setenv("WALKBUDDY_ML_MOCK", "1")
    client = TestClient(_build_app(yolo=None))
    resp = client.get("/ml/model-info")
    assert resp.status_code == 200
    body = resp.json()
    assert body["model"] == "walkbuddy-yolo-mock"
    assert body["class_count"] == 8
    assert body["mock"] is True


## 5. File I added: `requirements-dev.txt`

When I first tried to run my tests, I found `pytest` wasn't actually installed in the project's virtual environment, even though the existing tests assume it. That means CI wouldn't be able to run them either. So I added a `requirements-dev.txt` for test-only dependencies.

I kept it simple: it uses `-r requirements.txt` to pull in all the normal runtime deps (which already include `httpx`, the thing FastAPI's `TestClient` needs), and then adds `pytest` pinned to the version I verified with. This matches the repo's "one pinned requirements file" style but keeps test-only stuff separate from what the server needs to run in production.

In [ ]:
# requirements-dev.txt

# Test-only dependencies (not needed at runtime).
# Install with:  pip install -r requirements-dev.txt
#
# Runtime deps (including httpx, which FastAPI's TestClient uses) live in
# requirements.txt and are pulled in via the -r line below.
-r requirements.txt

pytest==9.1.1


## 6. File I added: `docs/ml_inference.md` (summary)

I wrote a markdown doc so anyone (frontend devs, reviewers) can understand and use the endpoint without reading the code. In my own words, it covers:

- **What it is** — a pure addition that serves the model through the existing pipeline, doesn't change any existing route.
- **Design** — how it reuses `app.state.yolo`, the limiter, `vision_adapter`, and the guidance helpers, and how `model`/`classes` come from `yolo.names`.
- **Mock mode** — the `WALKBUDDY_ML_MOCK` flag, that it's off by default, works with no weights, and returns `walkbuddy-yolo-mock`.
- **The two endpoints** — `POST /ml/navigate` and `GET /ml/model-info`, each with an example JSON response.
- **Error handling** — the 503 (no model) and 500 (inference failed) responses, and temp-file cleanup.
- **The 7-vs-8 class lineage note** — explains the config-vs-weights mismatch and that the endpoint just reports whatever the model actually has.
- **How to run the tests** — install `requirements-dev.txt`, then `pytest`.

The shell cell below shows the quick-start bit from that doc (how to flip on mock mode).

In [ ]:
# Mock mode: accepts 1/true/yes/on; unset (or anything else) = off
export WALKBUDDY_ML_MOCK=1

# Now /ml/navigate and /ml/model-info return a deterministic fake result
# in the exact same contract, with NO weights loaded and NO inference.
# The "model" field is "walkbuddy-yolo-mock" so the frontend knows it's fake.


## 7. How I tested it

I tested two ways: automated unit tests (the important one for CI) and manual `curl` commands against a running server.

**Automated:** I ran the 9 tests from the backend directory. They all pass and, crucially, they don't need the real `best.pt` weights — they use the fake model + monkeypatched adapter I described earlier.

**Manual:** if the server is running (`python main.py`), I can hit the endpoints with `curl`. In mock mode I don't even need weights. (If `WALKBUDDY_API_KEY` is set on the server, add `-H "X-API-Key: $WALKBUDDY_API_KEY"`.)

The commands I used are in the shell cell below, and the real pytest output is in the cell after it.

In [ ]:
# --- Automated tests (from the backend directory) ---
pip install -r requirements-dev.txt
pytest tests/test_ml_inference.py -v

# --- Manual smoke test with curl (server must be running) ---

# Ask which classes the model knows
curl http://localhost:8000/ml/model-info

# Send a frame and get detections + guidance
curl -X POST http://localhost:8000/ml/navigate \
  -F "file=@frame.jpg"

# Same, but in mock mode (no weights required) — start the server with the flag:
#   WALKBUDDY_ML_MOCK=1 python main.py
curl -X POST http://localhost:8000/ml/navigate -F "file=@frame.jpg"
# -> {"model":"walkbuddy-yolo-mock", "classes":[...8...], "detections":[{"category":"table",...}], ...}


This is the actual output I got — all 9 tests passing:

```
============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- .../.venv/bin/python
cachedir: .pytest_cache
rootdir: .../software_side/walkbuddy_reactNative/backend
plugins: anyio-4.12.1
collecting ... collected 9 items

tests/test_ml_inference.py::test_navigate_contract_eight_class PASSED    [ 11%]
tests/test_ml_inference.py::test_navigate_contract_seven_class PASSED    [ 22%]
tests/test_ml_inference.py::test_navigate_empty_file_short_circuits PASSED [ 33%]
tests/test_ml_inference.py::test_navigate_503_when_model_unavailable PASSED [ 44%]
tests/test_ml_inference.py::test_navigate_500_on_adapter_error PASSED    [ 55%]
tests/test_ml_inference.py::test_model_info_reports_active_classes PASSED [ 66%]
tests/test_ml_inference.py::test_model_info_503_when_model_unavailable PASSED [ 77%]
tests/test_ml_inference.py::test_navigate_mock_mode_no_weights PASSED    [ 88%]
tests/test_ml_inference.py::test_model_info_mock_mode_no_weights PASSED  [100%]

============================== 9 passed in 3.36s ===============================
```

## 8. What I learned / next steps

**What I learned:**

- **Reuse beats rewriting.** The biggest win was realising the model, limiter, adapter, and guidance helpers already existed. My job was really just to wire them into a clean new contract, not to write inference code. Reading the existing code first saved me a lot of work and kept behaviour consistent.
- **Don't hardcode data that the model already carries.** Reading class names from `yolo.names` at runtime is what makes the endpoint survive the 7-vs-8 class confusion. If I'd typed the class list into my code, it would silently be wrong for one of the two models.
- **Mock mode is genuinely useful.** Being able to develop the API and frontend before real weights exist (and to run tests with no weights) removes a big blocker for the team.
- **Match the existing error style.** Copying the 503/500 patterns and the temp-file `finally` cleanup means my endpoint behaves like the rest of the backend, so there are no surprises.

**Next steps / follow-ups (out of scope for this task):**

- **Resolve the 7-vs-8 class lineage mismatch.** The config says 8 classes (`couch`) but the verified `best.pt` has 7. Someone needs to confirm which model is the real production one and line up the config/weights. My endpoint already reports whatever is loaded, so it won't break either way — but the underlying mismatch should still be fixed.
- **Real latency numbers.** Right now `inference_time_ms` is measured but I haven't benchmarked it against actual weights on the target hardware.
- **Maybe add distance estimation.** The contract has room for it (the guidance events carry `distance_m`), but that's a separate feature.
- **Consider adding `pytest` to CI config** so these tests actually run on every PR now that `requirements-dev.txt` exists.